# Classification of single-cell SMAD time courses

This notebook compares control, high-dose TGF-beta and high-dose GDF11 trajectories. It deliberately contains only experiment orchestration and interpretation. Data import, model definition and optimization live in reusable modules under `src/`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import train_test_split

repository_root = Path.cwd()
if not (repository_root / "src").is_dir():
    repository_root = repository_root.parent
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

from src.io import load_smad_timecourses
from src.models.timecourse_classifyer import TimecourseClassifier
from src.train_torch import TrainingConfig, train

## Data and target representation

`load_smad_timecourses()` returns arrays in **cells × timepoints** orientation. PyTorch `Conv1d` expects **samples × channels × timepoints**, so a singleton channel axis is added below.

The targets are zero-based integer indices rather than one-hot vectors. This matches `CrossEntropyLoss`, which combines log-softmax with negative log-likelihood and therefore expects raw logits plus one class index per sample.

In [ ]:
CLASS_NAMES = ("control", "TGF-beta high", "GDF11 high")

smad_data = load_smad_timecourses()
times = smad_data.times
conditions = (
    smad_data.control,
    smad_data.tgfb_high,
    smad_data.gdf11_high,
)

features = np.concatenate(conditions, axis=0).astype(np.float32)
features = features[:, np.newaxis, :]
labels = np.concatenate(
    [
        np.full(len(condition), class_index, dtype=np.int64)
        for class_index, condition in enumerate(conditions)
    ]
)

In [ ]:
print(f"features: {features.shape}  (samples, channels, timepoints)")
for class_index, class_name in enumerate(CLASS_NAMES):
    sample_count = np.count_nonzero(labels == class_index)
    print(f"{class_name:15s}: {sample_count:4d} cells")

## Condition-level trajectory summaries

The median and interquartile range reveal which differences are already visible at population level. They do not describe the full single-cell distribution: two conditions can have similar medians while differing in response timing, transient bursts or heterogeneous subpopulations.

In [ ]:
figure, axes = plt.subplots(1, len(CLASS_NAMES), figsize=(10, 3), sharey=True)

for class_index, (axis, class_name) in enumerate(zip(axes, CLASS_NAMES)):
    trajectories = features[labels == class_index, 0, :]
    lower, median, upper = np.quantile(
        trajectories,
        (0.25, 0.5, 0.75),
        axis=0,
    )
    axis.fill_between(times, lower, upper, color="black", alpha=0.25)
    axis.plot(times, median, color="tab:blue", linewidth=2)
    axis.set_title(class_name)
    axis.set_xlabel("time")

axes[0].set_ylabel("normalized SMAD signal")
figure.tight_layout()

## Held-out test split

The test set is separated before training and is not used for early stopping. Stratification preserves the class proportions in both partitions.

This is a random split at cell level and therefore measures generalization to unseen cells from the same dataset. If cells originate from distinct experiments, plates or biological replicates, a grouped split by replicate would provide a stricter estimate of cross-experiment generalization.

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    features,
    labels,
    test_size=0.25,
    random_state=42,
    stratify=labels,
)

x_train_tensor = torch.from_numpy(x_train)
y_train_tensor = torch.from_numpy(y_train)
x_test_tensor = torch.from_numpy(x_test)

print(f"training pool: {x_train.shape}")
print(f"held-out test: {x_test.shape}")

## Model and optimization

The classifier has two convolutional branches. One processes the original signal and can learn absolute levels or slowly varying temporal motifs. The second processes first differences and is more sensitive to transitions while suppressing constant baseline offsets. Their features are concatenated before the dense classifier.

The model returns logits without a final softmax. Optimization uses Adam, exponential learning-rate decay and early stopping on validation loss. The training function restores the parameters from the epoch with the lowest validation loss.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TimecourseClassifier(
    input_length=x_train.shape[-1],
    num_classes=len(CLASS_NAMES),
    dropout=0.15,
)
training_config = TrainingConfig(
    epochs=100,
    batch_size=512,
    validation_fraction=0.25,
    learning_rate=1e-3,
    learning_rate_decay=0.92,
    patience=10,
    report_every=5,
    random_seed=42,
)

print(f"training on {device}")
history = train(
    model,
    x_train_tensor,
    y_train_tensor,
    config=training_config,
    device=device,
)
print(f"best epoch: {history.best_epoch}")

## Training diagnostics

A widening gap between training and validation curves indicates overfitting. A plateau in both curves can instead reflect exhausted model capacity, an overly small learning rate or irreducible overlap between the three cellular response distributions.

In [ ]:
epochs = np.arange(1, len(history.training_loss) + 1)
figure, axes = plt.subplots(1, 2, figsize=(9, 3))

axes[0].plot(epochs, history.training_loss, label="training")
axes[0].plot(epochs, history.validation_loss, label="validation")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("cross-entropy loss")
axes[0].legend()

axes[1].plot(epochs, history.training_accuracy, label="training")
axes[1].plot(epochs, history.validation_accuracy, label="validation")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("accuracy")
axes[1].legend()

figure.tight_layout()

## Evaluation on the held-out cells

The confusion matrix shows whether errors are symmetric or concentrated between biologically similar ligand responses. Per-class precision and recall are more informative than accuracy alone because the three conditions contain different numbers of cells.

In [ ]:
model.eval()
with torch.no_grad():
    test_logits = model(x_test_tensor.to(device))
    test_predictions = test_logits.argmax(dim=1).cpu().numpy()

test_accuracy = np.mean(test_predictions == y_test)
print(f"test accuracy: {test_accuracy:.1%}\n")
print(
    classification_report(
        y_test,
        test_predictions,
        target_names=CLASS_NAMES,
        zero_division=0,
    )
)

ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_predictions,
    display_labels=CLASS_NAMES,
    cmap="Blues",
)

## Save the trained state

The checkpoint stores the architectural metadata needed to reconstruct the model together with the restored best parameter state. Training history is kept separate because it describes this run rather than the inference model.

In [ ]:
checkpoint_path = repository_root / "timecourse_classifier.pth"
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "input_length": model.input_length,
        "class_names": CLASS_NAMES,
    },
    checkpoint_path,
)
print(f"saved checkpoint to {checkpoint_path}")